# Fantrax API Wrapper: Comprehensive Feature Guide & Analysis Notebook

This notebook demonstrates how to access and analyze Fantrax Fantasy Premier League data using the clean, modular  package.

### Capabilities Covered:
1. **Environment Configuration**: Loading credentials and league settings from .
2. **League & Schedule Queries**: Fetching head-to-head weekly matchups and cumulative standings.
3. **Roster Management**: Extracting starting XIs, bench reserves, and injured reserve for any team.
4. **Player Database & Box Scores**: Querying player stats, historical game logs, and latest news.
5. **Live Scoring & Projections**: Fetching live scoring and projected points per week.
6. **Position-Specific Scoring Engine**: Calculating FPts and Adjusted FPts (aFPts) across positions (G, D, M, F).
7. **Manager Efficiency Analytics**: Optimal lineup solver, lineup inefficiency, and standings against weekly median.
8. **Waiver Wire Evaluation**: Filtering and ranking available free agents.
9. **Stat Normalization**: Per-90 stat metrics and trimmed mean opponent baselines.


## 1. Setup & Environment Configuration
Sourcing environment variables from  and initializing .

In [1]:
import os
import sys
import importlib
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Ensure parent directory is in Python path
sys.path.insert(0, os.path.abspath(".."))

import fantrax_wrapper
import fantrax_wrapper.models
import fantrax_wrapper.models.team
import fantrax_wrapper.client
import fantrax_wrapper.rpc
import fantrax_wrapper.analytics
import fantrax_wrapper.scoring

importlib.reload(fantrax_wrapper.models.team)
importlib.reload(fantrax_wrapper.models)
importlib.reload(fantrax_wrapper.rpc)
importlib.reload(fantrax_wrapper.scoring)
importlib.reload(fantrax_wrapper.analytics)
importlib.reload(fantrax_wrapper.client)
importlib.reload(fantrax_wrapper)

from fantrax_wrapper import (
    FantraxClient,
    FantasyScoringEngine,
    calculate_manager_efficiency,
    calculate_median_standings,
    evaluate_waiver_candidates,
    normalize_per_90_stats,
    compute_positional_opponent_averages,
    load_env_file,
)

# Load environment variables from soccer/config/.env
env_loaded = load_env_file("../config/.env")
print(f"Loaded environment file: {env_loaded}")

cookie_path = "fantraxloggedin.cookie"
client = FantraxClient(cookie_path=cookie_path, auto_login=True)
print(f"FantraxClient initialized for League ID: {client.league_id}")
print(f"Client has get_draft_rankings: {hasattr(client, 'get_draft_rankings')}")


Loaded environment file: True
FantraxClient initialized for League ID: de19pr8smd5cytbm
Client has get_draft_rankings: True


## 2. Matchup Schedule & League Table
Fetching head-to-head match schedules, weekly scores, and calculating official league table standings.

In [2]:
# Fetch weekly head-to-head matchups
matchups_df = client.get_matchups(through_week=38, as_dataframe=True)
print(f"Fetched {len(matchups_df)} matchup rows.")
matchups_df.head(10)


Fetched 532 matchup rows.


,week,team,team_id,score,opponent,opponent_score,matchupId,result
0,1,Arnie-senal,mqmbxwrtmdg6mr2l,83.00,FC VAR,80.50,0,W
1,1,FC VAR,1t98phesmd5cytbv,80.50,Arnie-senal,83.00,0,L
2,1,Cold FC,ja8ivcs5mdew697k,123.75,youngmoon,66.25,1,W
3,1,youngmoon,s5ksmx70mdmj1kxc,66.25,Cold FC,123.75,1,L
4,1,WayneRooney10,cvw1xxnsmdg4tqui,102.75,sduvuuru,77.75,2,W
5,1,sduvuuru,p7p53wgimdj1ldb3,77.75,WayneRooney10,102.75,2,L
6,1,FC Purulona,1kg0cgl2mdmidwrt,152.75,Wallalujah FC,131.75,3,W
7,1,Wallalujah FC,af1qhccfmdhlee7c,131.75,FC Purulona,152.75,3,L
8,1,Toadenham Frogspur,39gqvc1wmdg4tnfu,120.50,Estimated Profit,50.00,4,W
9,1,Estimated Profit,hzn5pbmkmdewwr84,50.00,Toadenham Frogspur,120.50,4,L


In [3]:
# Fetch overall league table standings
standings_df = client.get_league_table(through_week=38)
print("Official League Table:")
standings_df


Official League Table:


,wins,draws,losses,record,points,fpts_for,fpts_against,avg_fpts_for,avg_fpts_against
team,,,,,,,,,
FC VAR,27,0,11,27-0-11,81,3489.00,2873.00,91.82,75.61
Toadenham Frogspur,24,1,13,24-1-13,73,3196.50,2943.50,84.12,77.46
youngmoon,22,0,15,22-0-15,66,3140.25,3057.25,84.87,82.63
Traderjoe18,21,0,17,21-0-17,63,3310.50,3149.00,87.12,82.87
Arnie-senal,21,0,17,21-0-17,63,2860.50,2892.50,75.28,76.12
Smip Estonian,20,0,18,20-0-18,60,3330.75,3104.50,87.65,81.70
Wallalujah FC,19,0,19,19-0-19,57,3420.00,3236.75,90.00,85.18
FC Purulona,19,0,19,19-0-19,57,3224.00,3163.00,84.84,83.24
sduvuuru,19,0,19,19-0-19,57,3031.75,3161.75,79.78,83.20


## 3. Team Roster Breakdown
Extracting team roster breakdown (Starting XI, Reserves, Injured Reserve) for a given week.

In [4]:
# Fetch roster for a team (using first team from matchups)
sample_team_id = matchups_df["team_id"].iloc[0] if not matchups_df.empty else ""
if sample_team_id:
    roster = client.get_team_roster(team_id=sample_team_id, period=1)
    print(f"Roster for Team ID {sample_team_id} (Week 1):")
    print(f"Goalkeeper: {roster.goalkeeper.player_name if roster.goalkeeper else 'None'}")
    print(f"Defenders: {[d.player_name for d in roster.defenders]}")
    print(f"Midfielders: {[m.player_name for m in roster.midfielders]}")
    print(f"Forwards: {[f.player_name for f in roster.forwards]}")
    print(f"Reserves (Bench): {[r.player_name for r in roster.reserves]}")
    print(f"Injured Reserve (IR): {[i.player_name for i in roster.injured_reserve]}")


Roster for Team ID mqmbxwrtmdg6mr2l (Week 1):
Goalkeeper: Alisson Becker
Defenders: ['Daniel Munoz', 'Chris Richards', 'William Saliba']
Midfielders: ['Harvey Barnes', 'Jeremy Doku', 'Youri Tielemans', 'Mats Wieffer']
Forwards: ['Niclas Füllkrug', 'Callum Hudson-Odoi', 'Ollie Watkins']
Reserves (Bench): ['Tosin Adarabioyo', 'Conor Bradley', 'Jean-Ricner Bellegarde', 'John McGinn', 'Marc Guiu']
Injured Reserve (IR): ['Vitalii Mykolenko']


## 4. Player Database, Game Logs & News
Querying the player database, inspecting detailed game logs for individual players, and fetching news updates.

In [5]:
# Fetch top players database
players_df = client.get_all_players(as_dataframe=True)
print(f"Total players fetched: {len(players_df)}")
players_df[["name", "teamShortName", "posShortNames", "fpts", "fpts_per_game", "next_match"]].head(10)


Total players fetched: 920


,name,teamShortName,posShortNames,fpts,fpts_per_game,next_match
0,Bruno Fernandes,MUN,M,633.5,18.10,MUN 3<br/>@BHA 0 F
1,Erling Haaland,MCI,F,509.0,14.54,AVL 2<br/>@MCI 1 F
2,Antoine Semenyo,BOU/MCI,F,444.0,12.00,AVL 2<br/>@MCI 1 F
3,Igor Thiago,BRF,F,431.0,11.34,BRF 1<br/>@LIV 1 F
4,Bukayo Saka,ARS,M,411.0,13.26,ARS 2<br/>@CRY 1 F
5,Dominik Szoboszlai,LIV,M,409.0,11.36,BRF 1<br/>@LIV 1 F
6,Jarrod Bowen,WHU,F,404.0,10.63,LEE 0<br/>@WHU 3 F
7,Morgan Gibbs-White,NOT,M,403.5,10.91,BOU 1<br/>@NOT 1 F
8,Declan Rice,ARS,M,395.0,10.97,ARS 2<br/>@CRY 1 F
9,Elliot Anderson,NOT,M,387.5,10.20,BOU 1<br/>@NOT 1 F


In [6]:
# Query game log for a top player
if not players_df.empty:
    sample_player_id = players_df.iloc[0]["scorerId"]
    sample_player_name = players_df.iloc[0]["name"]
    print(f"Fetching game log for: {sample_player_name} (ID: {sample_player_id})")
    game_log_df = client.get_player_game_log(player_id=sample_player_id, year=2025)
    game_log_df.head(10)

Fetching game log for: Bruno Fernandes (ID: 05gcr)


In [7]:
# Fetch latest news for a top player
if not players_df.empty:
    sample_player_id = players_df.iloc[0]["scorerId"]
    sample_player_name = players_df.iloc[0]["name"]
    news = client.get_player_news(player_id=sample_player_id)
    print(f"Latest News for {sample_player_name}:")
    print(news if news else "No recent news analysis text.")

Latest News for Bruno Fernandes:
Fernandes contributed one assist across five appearances for Portugal at the World Cup, which ended without a deep run for the national team, before returning to resume his role as United's creative hub after a season in which he logged 3,066 minutes across 35 Premier League starts and produced nine goals and 21 assists, an assist tally among the league's best backed by 136 chances created and penalty duty. Under coach Michael Carrick, he is expected to remain the focal point of United's attack, while continuing to build up his fitness as preseason progresses.


## 5. Live Scoring & Projections
Fetching live scoring stats and projected fantasy points map for a scoring period.

In [8]:
# Fetch live scoring & projections for Week 1
live_scoring_df = client.get_live_scoring(period=1)
print(f"Fetched {len(live_scoring_df)} projection entries for Week 1:")
live_scoring_df.head(10)


Fetched 146 projection entries for Week 1:


,teamId,status,scorerId,fantrax_projection,week
0,8j17gbjmmdewvf8d,ACTIVE,04tig,18.50,1
1,8j17gbjmmdewvf8d,ACTIVE,04qrc,-0.50,1
2,8j17gbjmmdewvf8d,ACTIVE,02m5b,23.50,1
3,8j17gbjmmdewvf8d,ACTIVE,06ymu,9.00,1
4,8j17gbjmmdewvf8d,ACTIVE,05s08,13.75,1
5,8j17gbjmmdewvf8d,ACTIVE,05r8o,4.50,1
6,8j17gbjmmdewvf8d,ACTIVE,06cms,8.50,1
7,8j17gbjmmdewvf8d,ACTIVE,05rb8,8.00,1
8,8j17gbjmmdewvf8d,ACTIVE,06p4k,13.50,1
9,8j17gbjmmdewvf8d,ACTIVE,05gcr,12.50,1


## 6. Position-Specific Fantasy Scoring Engine
Calculating fantasy points (FPts) and adjusted fantasy points (aFPts) using position-specific scoring rules (GK vs D, M, F).

In [9]:
scoring_engine = FantasyScoringEngine()

# Sample box score lines
midfielder_line = {"G": 1, "AT": 1, "KP": 3, "SOT": 2, "TKW": 2, "DIS": -1, "YC": -2}
goalkeeper_line = {"CS": 1, "SV": 4, "GAO": 0, "PKS": 0, "DIS": 0}

mid_fpts = scoring_engine.calculate_fantasy_score(midfielder_line, position="M")
mid_afpts = scoring_engine.calculate_adjusted_fantasy_score(midfielder_line, position="M")

gk_fpts = scoring_engine.calculate_fantasy_score(goalkeeper_line, position="G")
gk_afpts = scoring_engine.calculate_adjusted_fantasy_score(goalkeeper_line, position="G")

print(f"Midfielder Line: {midfielder_line}")
print(f"-> FPts: {mid_fpts} | aFPts (non-luck): {mid_afpts}")
print(f"Goalkeeper Line: {goalkeeper_line}")
print(f"-> FPts: {gk_fpts} | aFPts (non-luck): {gk_afpts}")


Midfielder Line: {'G': 1, 'AT': 1, 'KP': 3, 'SOT': 2, 'TKW': 2, 'DIS': -1, 'YC': -2}
-> FPts: 31.5 | aFPts (non-luck): 16.5
Goalkeeper Line: {'CS': 1, 'SV': 4, 'GAO': 0, 'PKS': 0, 'DIS': 0}
-> FPts: 16.0 | aFPts (non-luck): 16.0


## 7. Manager Efficiency & Optimal Lineup Analytics
Solving for optimal starting XIs (), measuring lineup inefficiency, and computing standings against weekly median score.

In [10]:
if not matchups_df.empty:
    # Simulate max points for demonstration if missing
    demo_matchups = matchups_df.copy()
    demo_matchups["max_points_for"] = demo_matchups["score"] + 15.0
    
    efficiency_df = calculate_manager_efficiency(demo_matchups)
    print("Manager Lineup Efficiency Metrics:")
    efficiency_df

Manager Lineup Efficiency Metrics:


In [11]:
if not matchups_df.empty:
    median_standings_df = calculate_median_standings(demo_matchups)
    print("Standings Evaluated Against Weekly League Median:")
    median_standings_df

Standings Evaluated Against Weekly League Median:


## 8. Waiver Wire Candidate Evaluation
Filtering available free agents by minutes played, average FPts per game, and position.

In [12]:
if not players_df.empty:
    # Mock rostered player list
    rostered_names = players_df["name"].iloc[:10].tolist()
    
    waivers_df = evaluate_waiver_candidates(
        df_player_stats=players_df,
        rostered_player_names=rostered_names,
        current_week=38,
        min_last_week_minutes=0,
        exclude_goalkeepers=True
    )
    print(f"Evaluated {len(waivers_df)} free agent waiver candidates:")
    waivers_df[["name", "teamShortName", "posShortNames", "fpts", "fpts_per_game"]].head(10)


Evaluated 910 free agent waiver candidates:


## 9. Stat Normalization & Positional Opponent Averages
Normalizing counting statistics to per-90 metrics and computing trimmed mean positional baselines against opponents.

In [13]:
# Sample player match performance rows
sample_perf = pd.DataFrame([
    {"player_name": "Player A", "position": "M", "opp": "ARS", "minutes": 90, "kp": 4, "sot": 2, "g": 1},
    {"player_name": "Player B", "position": "M", "opp": "ARS", "minutes": 45, "kp": 2, "sot": 1, "g": 0},
    {"player_name": "Player C", "position": "D", "opp": "BRE", "minutes": 90, "kp": 1, "sot": 0, "g": 0},
])

norm_df = normalize_per_90_stats(sample_perf, stat_columns=["kp", "sot", "g"], minutes_col="minutes")
print("Per-90 Normalized Performance Stats:")
norm_df

Per-90 Normalized Performance Stats:


,player_name,position,opp,minutes,kp,sot,g,kp_per_min,kp_per_90,sot_per_min,sot_per_90,g_per_min,g_per_90
0,Player A,M,ARS,90,4,2,1,0.044,3.96,0.022,1.98,0.011,0.99
1,Player B,M,ARS,45,2,1,0,0.044,3.96,0.022,1.98,0.000,0.00
2,Player C,D,BRE,90,1,0,0,0.011,0.99,0.000,0.00,0.000,0.00


In [14]:
opp_averages_df = compute_positional_opponent_averages(norm_df, per90_columns=["kp_per_90", "sot_per_90"])
print("Positional Opponent Trimmed Mean Averages:")
opp_averages_df

Positional Opponent Trimmed Mean Averages:


,opp,position,positional_avg_kp_per_90,positional_avg_sot_per_90
0,ARS,M,3.96,1.98
1,BRE,D,0.99,0.00


## 10. Raw League Metadata & Custom RPC Queries
`FantraxClient.get_raw_league_info()` provides low-level access to any Fantrax RPC method (`getLeagueInfo`, `getLeagueRules`, `getDraftResults`, `getTransactionHistory`).

In [15]:
# Query raw league metadata and rules
raw_info = client.get_raw_league_info(method="getLeagueInfo")
league_name = raw_info.get("name", "Fantrax Premier League")
print(f"Raw League Name: {league_name}")

# Query raw draft results / transactions
draft_info = client.get_raw_league_info(method="getDraftResults")
print(f"Draft Results payload keys: {list(draft_info.keys())[:5] if isinstance(draft_info, dict) else type(draft_info)}")


Raw League Name: Fantrax Premier League


Draft Results payload keys: ['availableDraftTypes', 'historyLeagues', 'draftPicksOrdered', 'fantasyTeamsOrdered', 'draftType']


## 11. Custom Fantasy Scoring Engine Configuration
The `FantasyScoringEngine` supports custom positional point allocations per stat event (Goal, Assist, Clean Sheet, Tackle, Interception, Save, Key Pass).

In [16]:
from fantrax_wrapper.scoring import FantasyScoringEngine

# Initialize FantasyScoringEngine with custom positional scoring weights
custom_scoring_weights = {
    "D": {"G": 10.0, "AT": 7.0, "CS": 6.0, "TKW": 1.0, "KP": 2.0, "INT": 1.0},
    "M": {"G": 9.0, "AT": 7.0, "TKW": 1.0, "KP": 2.0, "INT": 1.0},
    "F": {"G": 8.0, "AT": 7.0, "KP": 2.0, "SOT": 2.0},
}

custom_engine = FantasyScoringEngine(custom_rules=custom_scoring_weights)
sample_stats = {"G": 1, "AT": 1, "KP": 3, "TKW": 2}
score_m = custom_engine.calculate_fantasy_score(sample_stats, position="M")
print("Custom Fantasy Scoring Engine Initialized.")
print(f"Calculated Midfielder Score for stats {sample_stats}: {score_m} pts")


Custom Fantasy Scoring Engine Initialized.
Calculated Midfielder Score for stats {'G': 1, 'AT': 1, 'KP': 3, 'TKW': 2}: 24.0 pts


## 12. Data Export & CSV / JSON Integration
All tabular data returned by `FantraxClient` can be exported directly to CSV or JSON for downstream modeling.

In [17]:
# Export league standings to CSV
standings_df = client.get_league_table()
if not standings_df.empty:
    csv_path = "league_standings.csv"
    standings_df.to_csv(csv_path, index=True)
    print(f"Exported league standings to {csv_path}")

# Export top players to JSON
players_df = client.get_all_players(as_dataframe=True)
if not players_df.empty:
    json_path = "top_players.json"
    players_df.head(20).to_json(json_path, orient="records", indent=2)
    print(f"Exported top 20 players to {json_path}")


Exported league standings to league_standings.csv


Exported top 20 players to top_players.json


## 13. Optimal Lineup Optimizer (Given Team & Matchweek)
`FantraxClient.get_optimal_lineup(team_id, period)` calculates the maximum possible Starting XI fantasy score and individual player lineup selection for any team in a given week.

In [18]:
# Calculate optimal lineup for team mqmbxwrtmdg6mr2l in Week 1
matchups_df = client.get_matchups(through_week=38, as_dataframe=True)
if not matchups_df.empty:
    sample_team_id = matchups_df["team_id"].iloc[1]
    optimal_info = client.get_optimal_lineup(team_id=sample_team_id, period=10)
    
    print(f"Optimal Lineup Analysis for Team {sample_team_id} (Week 1):")
    print(f"  Actual Score Achieved: {optimal_info['actual_score']} pts")
    print(f"  Max Optimal Score Possible: {optimal_info['optimal_score']} pts")
    print(f"  Manager Lineup Efficiency: {optimal_info['efficiency_pct']}%\n")
    
    starters_df = pd.DataFrame(optimal_info["starters"])
    print(f"Optimal Starting XI ({len(starters_df)} players):")
    starters_df[["player_name", "position", "score"]]


Optimal Lineup Analysis for Team 1t98phesmd5cytbv (Week 1):
  Actual Score Achieved: 821.0 pts
  Max Optimal Score Possible: 879.25 pts
  Manager Lineup Efficiency: 93.38%

Optimal Starting XI (11 players):


## 14. Draft Rankings & Player ADP Projections
`FantraxClient.get_draft_rankings(season_or_projection, league_id)` pulls the complete draft rankings list, projected fantasy points, points per game, and percentage drafted.

In [19]:
# Fetch draft rankings list
draft_ranks_df = client.get_draft_rankings(season_or_projection="PROJECTION_0_926_SEASON", league_id="iku2an1nmryzcfa2")

print(f"Fetched {len(draft_ranks_df)} draft ranked players:")
draft_ranks_df.head(15)


Fetched 698 draft ranked players:


,rank,player_id,name,short_name,team,team_name,position,adp,total_pts,projected_pts,pts_per_game,percent_drafted,adp_rank
0,60,061vq,Erling Haaland,E. Haaland,MCI,Manchester City,F,1.50,464.0,464.0,15.47,100.00,1
1,46,05gcr,Bruno Fernandes,Bruno Fernandes,MUN,Manchester United,M,2.06,480.0,480.0,15.00,100.00,2
2,353,04y92,Bukayo Saka,B. Saka,ARS,Arsenal,M,3.22,500.5,500.5,15.64,100.00,3
3,692,068y0,Antoine Semenyo,A. Semenyo,MCI,Manchester City,F,4.81,396.5,396.5,13.22,100.00,4
4,624,06cmm,Dominik Szoboszlai,D. Szoboszlai,LIV,Liverpool,M,6.50,378.0,378.0,11.81,100.00,5
5,649,06v21,Florian Wirtz,F. Wirtz,LIV,Liverpool,M,7.44,413.5,413.5,13.34,100.00,6
6,553,067yz,Matheus Cunha,Matheus Cunha,MUN,Manchester United,F,12.09,385.0,385.0,12.42,100.00,7
7,584,06v07,Rayan Cherki,R. Cherki,MCI,Manchester City,M,12.16,389.0,389.0,13.89,100.00,8
8,326,04agz,Declan Rice,D. Rice,ARS,Arsenal,M,12.41,340.5,340.5,10.32,100.00,9
9,416,04qr9,Morgan Gibbs-White,M. Gibbs-White,NOT,Nottingham Forest,M,12.50,347.5,347.5,10.53,100.00,10
